# Atık Tespiti — YOLOv8 Eğitimi (6 Kategori)

Bu notebook, drone fotoğraflarındaki çöpü **kağıt, cam, metal, plastik, geri dönüştürülemez, organik** olmak üzere 6 kategoride tespit eden bir YOLOv8 (object detection, bounding-box) modeli eğitir.

**Veri seti konumu:** `Benimle Paylaşılanlar / Drone-proje / dataset / goruntuler_veriseti` — içinde `data.yaml`, `train/`, `valid/`, `test/` klasörleri ve `2026 ocak tüm veriseti` bulunuyor (standart Roboflow/YOLO export formatı). Ham sınıf adlarınız 6 Türkçe kategoriyle birebir aynı olmak zorunda değil — `waste-detection-service/class_map.py` ham isimleri bu 6 kategoriye eşliyor; yeni ham isimler kullandıysanız (ör. organik için) o dosyadaki `RAW_TO_CATEGORY` sözlüğüne ekleyin.

**'Benimle Paylaşılanlar' klasörleri Colab'da otomatik görünmez** — `drive.mount()` sadece kendi Drive'ınızı (`MyDrive`) bağlar. `dataset` klasörüne Drive arayüzünde bir kez **'Kısayol ekle' / 'Add shortcut to Drive'** yaptıysanız (kendi `Drive'ım` köküne kısayol olarak eklendiyse) aşağıdaki §2 hücresi `data.yaml`'ı otomatik bulur; henüz yapmadıysanız o adımı önce tamamlayın.

**Çıktı:** eğitilmiş `best.pt` — bunu indirip `waste-detection-service/model/best.pt` konumuna koyun ve servisi yeniden başlatın.

## 1) Kurulum + Drive bağlama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

## 2) Veri seti yolunu bul

`DATASET_YAML`'ı boş bırakırsanız, Drive'ınızda `goruntuler_veriseti/data.yaml`'ı otomatik arar (kısayolu nereye eklediğinizden bağımsız çalışır). Tam yolu zaten biliyorsanız aşağıya yazıp elle de belirtebilirsiniz.

In [ ]:
DATASET_YAML = ""  # boş = otomatik ara; biliyorsanız tam yolu buraya yazın, ör. "/content/drive/MyDrive/dataset/goruntuler_veriseti/data.yaml"

import glob
import os

if not DATASET_YAML:
    search_patterns = [
        "/content/drive/MyDrive/dataset/goruntuler_veriseti/data.yaml",  # dataset klasörü MyDrive kökünde kısayol ise
        "/content/drive/MyDrive/**/goruntuler_veriseti/data.yaml",       # kısayol başka bir yere eklendiyse
        "/content/drive/**/data.yaml",                                    # son çare: tüm Drive'da ara
    ]
    found = []
    for pattern in search_patterns:
        found = glob.glob(pattern, recursive=True)
        if found:
            break
    if not found:
        raise FileNotFoundError(
            "data.yaml Drive'ınızda bulunamadı.\n"
            "'Benimle Paylaşılanlar > Drone-proje > dataset' klasörüne Drive arayüzünde "
            "sağ tıklayıp 'Kısayol ekle' / 'Add shortcut to Drive' ile kendi Drive'ınıza "
            "ekleyin, sonra bu hücreyi tekrar çalıştırın."
        )
    DATASET_YAML = found[0]

assert os.path.exists(DATASET_YAML), f"data.yaml bulunamadı: {DATASET_YAML}"
print(f"✅ Kullanılacak data.yaml: {DATASET_YAML}")

## 3) Config — tek yerden değiştirin

In [ ]:
EPOCHS = 100
IMGSZ = 640
BASE_MODEL = "yolov8s.pt"  # hız/doğruluk dengesi: nano (yolov8n.pt) en hızlı ama daha az doğru;
                           # s en iyi denge (bkz. sohbet — çıkarım sunucu tarafında, uçuş sonrası
                           # toplu çalışıyor, gerçek zamanlı değil, bu yüzden doğruluğu önceliklendirdik);
                           # daha da yüksek doğruluk için: yolov8m.pt (daha yavaş eğitim + çıkarım)
RUN_NAME = "waste_detection_6class"

# Eğitim biter bitmez best.pt bu klasöre de kopyalanır (Colab oturumu kapanınca kaybolmasın diye)
# Varsayılan olarak data.yaml'ın bulunduğu klasörün yanına bir 'model_ciktilari' alt klasörü açar.
DRIVE_OUTPUT_DIR = os.path.join(os.path.dirname(DATASET_YAML), "model_ciktilari")

## 4) Veri seti sağlık kontrolü

`data.yaml`'daki sınıf isimlerini yazdırır — eğitimden önce bu isimlerin `class_map.py`'deki `RAW_TO_CATEGORY` sözlüğünde karşılığı olduğunu (veya olacağını) gözden geçirin.

In [ ]:
import yaml

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("Sınıflar (ham isimler):", data_cfg.get("names"))
print("Sınıf sayısı:", data_cfg.get("nc", len(data_cfg.get("names", []))))
print("\nBu isimlerin waste-detection-service/class_map.py -> RAW_TO_CATEGORY içinde")
print("6 kategoriden (kağıt/cam/metal/plastik/geri dönüştürülemez/organik) birine")
print("eşlendiğinden emin olun; eşlenmeyenler otomatik olarak 'geri dönüştürülemez' sayılır.")

## 5) Eğitim

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    name=RUN_NAME,
    patience=20,  # erken durdurma: 20 epoch boyunca iyileşme olmazsa dur
)

## 6) Doğrulama (validation seti üzerinde metrikler)

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Kategori bazlı mAP50:", dict(zip(data_cfg.get("names", []), metrics.box.maps)))

## 7) `best.pt`'yi Drive'a kopyala

In [ ]:
import shutil

best_pt_path = os.path.join("runs", "detect", RUN_NAME, "weights", "best.pt")
assert os.path.exists(best_pt_path), f"best.pt bulunamadı: {best_pt_path}"

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
dest_path = os.path.join(DRIVE_OUTPUT_DIR, "waste_detection_best.pt")
shutil.copy(best_pt_path, dest_path)

print(f"✅ Kopyalandı: {dest_path}")

## 8) Sıradaki adım

1. Yukarıdaki dosyayı (`waste_detection_best.pt`) Drive'ınızdan bilgisayarınıza indirin.
2. `best.pt` olarak yeniden adlandırıp `waste-detection-service/model/best.pt` konumuna koyun.
3. Servisi (yeniden) başlatın: `uvicorn app:app --host 127.0.0.1 --port 8000`
4. `GET /health` çağırıp `"ready": true` döndüğünü doğrulayın.